# VitroVision — Training Pipeline
Transfer Learning ด้วย `timm` (EfficientNet-B0) สำหรับ classify สถานะขวดเพาะเลี้ยงเนื้อเยื่อ

**Classes:** `healthy` | `contaminated` | `dead`  
**Input:** ภาพจาก VitroShelf ที่ `shelf_manager/data/**/*.jpg`  
**Output:** `models/final/classifier.pt`

In [ ]:
import os, sys, json, warnings
from pathlib import Path
from collections import Counter

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

warnings.filterwarnings('ignore')
print(f'torch: {torch.__version__} | timm: {timm.__version__} | device: cpu')

In [ ]:
# ── Config ──────────────────────────────────────────────
ROOT      = Path('..').resolve()
DATA_DIR  = ROOT / 'shelf_manager' / 'data'
MODEL_DIR = ROOT / 'models' / 'final'
MODEL_OUT = MODEL_DIR / 'classifier.pt'
CKPT_DIR  = ROOT / 'models' / 'checkpoints'

MODEL_DIR.mkdir(parents=True, exist_ok=True)
CKPT_DIR.mkdir(parents=True, exist_ok=True)

LABELS   = ['healthy', 'contaminated', 'dead']
L2I      = {l: i for i, l in enumerate(LABELS)}
IMG_SIZE = 224
BATCH    = 8
LR       = 1e-4
EPOCHS_HEAD  = 15   # Phase 1: train head เท่านั้น
EPOCHS_FULL  = 25   # Phase 2: fine-tune ทั้งหมด
DEVICE   = 'cpu'

print(f'DATA_DIR: {DATA_DIR}')
print(f'MODEL_OUT: {MODEL_OUT}')

## 1 — โหลด Dataset

In [ ]:
def load_dataset(data_dir):
    """อ่าน label จากชื่อไฟล์: S01-A-01_day000_0042_healthy.jpg → 'healthy'"""
    paths, labels = [], []
    for jpg in Path(data_dir).rglob('*.jpg'):
        label = jpg.stem.split('_')[-1]
        if label in L2I:
            paths.append(str(jpg))
            labels.append(label)
    return paths, labels

all_paths, all_labels = load_dataset(DATA_DIR)
counts = Counter(all_labels)
print(f'Dataset: {len(all_paths)} ภาพ')
for cls in LABELS:
    print(f'  {cls:15s}: {counts.get(cls, 0):4d} ภาพ')

if len(all_paths) < 30:
    print(f'\n⚠️  ภาพน้อยมาก ({len(all_paths)}) — model จะยังไม่แม่นยำ แนะนำ ≥200 ต่อ class')
    print('   pipeline นี้รันได้ เพื่อทดสอบโค้ดก่อนเก็บข้อมูลครบ')

In [ ]:
# แสดง sample ภาพ
fig, axes = plt.subplots(1, min(6, len(all_paths)), figsize=(14, 3))
if len(all_paths) == 1:
    axes = [axes]
for ax, path, label in zip(axes, all_paths[:6], all_labels[:6]):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.set_title(label, fontsize=9)
    ax.axis('off')
plt.suptitle('Sample images', fontsize=11)
plt.tight_layout()
plt.show()

## 2 — Train/Val Split

In [ ]:
# ถ้าข้อมูลน้อยเกินไปสำหรับ stratify → ใช้ random split
min_class_count = min(counts.get(c, 0) for c in LABELS if counts.get(c, 0) > 0)
use_stratify = min_class_count >= 3

train_paths, val_paths, train_labels, val_labels = train_test_split(
    all_paths, all_labels,
    test_size=0.2,
    stratify=all_labels if use_stratify else None,
    random_state=42,
)
print(f'Train: {len(train_paths)} | Val: {len(val_paths)}')
print(f'Train dist: {Counter(train_labels)}')
print(f'Val   dist: {Counter(val_labels)}')

## 3 — Augmentation

In [ ]:
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

train_aug = A.Compose([
    A.RandomResizedCrop(height=IMG_SIZE, width=IMG_SIZE, scale=(0.65, 1.0)),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.35, contrast=0.35, saturation=0.3, hue=0.08, p=0.8),
    A.GaussianBlur(blur_limit=(3, 7), p=0.3),
    A.GaussNoise(var_limit=(10.0, 50.0), p=0.3),
    A.RandomShadow(p=0.2),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=mean, std=std),
    ToTensorV2(),
])

# แสดง augmented samples
sample_img = cv2.cvtColor(cv2.imread(all_paths[0]), cv2.COLOR_BGR2RGB)
fig, axes = plt.subplots(1, 5, figsize=(14, 3))
for ax in axes:
    aug = A.Compose(train_aug.transforms[:-2])(image=sample_img)
    ax.imshow(aug['image']); ax.axis('off')
plt.suptitle('Augmented samples (same image)', fontsize=11)
plt.tight_layout(); plt.show()

## 4 — Dataset & DataLoader

In [ ]:
class BottleDataset(Dataset):
    def __init__(self, paths, labels, transform):
        self.paths     = paths
        self.labels    = [L2I[l] for l in labels]
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = cv2.imread(self.paths[idx])
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        out = self.transform(image=img)
        return out['image'], self.labels[idx]


train_ds = BottleDataset(train_paths, train_labels, train_aug)
val_ds   = BottleDataset(val_paths,   val_labels,   val_aug)

# เพิ่ม num_workers=0 บน Windows
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=0)

print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

## 5 — Model (EfficientNet-B0)

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=len(LABELS))
model = model.to(DEVICE)

total  = sum(p.numel() for p in model.parameters())
print(f'Parameters: {total:,}')
print(f'Classifier head: {model.classifier}')

In [ ]:
def set_trainable(model, backbone=False, head=True):
    """ควบคุมว่าจะ freeze/unfreeze ส่วนไหน"""
    for name, param in model.named_parameters():
        is_head = 'classifier' in name
        param.requires_grad = head if is_head else backbone
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f'Trainable params: {trainable:,}')

criterion = nn.CrossEntropyLoss()


def run_epoch(loader, optimizer=None, desc=''):
    training = optimizer is not None
    model.train() if training else model.eval()
    total_loss, correct, n = 0.0, 0, 0
    ctx = torch.enable_grad() if training else torch.no_grad()
    with ctx:
        for imgs, targets in loader:
            imgs, targets = imgs.to(DEVICE), targets.to(DEVICE)
            logits = model(imgs)
            loss   = criterion(logits, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(targets)
            correct    += (logits.argmax(1) == targets).sum().item()
            n          += len(targets)
    return total_loss / n, correct / n

## 6 — Phase 1: Train Head เท่านั้น

In [ ]:
set_trainable(model, backbone=False, head=True)
opt1   = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)
sched1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt1, T_max=EPOCHS_HEAD)

hist = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')

print(f'Phase 1 — {EPOCHS_HEAD} epochs (head only)')
for ep in range(1, EPOCHS_HEAD + 1):
    tl, ta = run_epoch(train_loader, opt1)
    vl, va = run_epoch(val_loader)
    sched1.step()
    hist['train_loss'].append(tl); hist['val_loss'].append(vl)
    hist['train_acc'].append(ta);  hist['val_acc'].append(va)
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model, CKPT_DIR / 'best_head.pt')
    print(f'  Ep {ep:2d}/{EPOCHS_HEAD} | loss {tl:.4f}/{vl:.4f} | acc {ta:.0%}/{va:.0%}')

## 7 — Phase 2: Fine-tune ทั้งหมด

In [ ]:
set_trainable(model, backbone=True, head=True)
opt2   = torch.optim.Adam(model.parameters(), lr=LR * 0.1)   # LR ต่ำกว่า Phase 1
sched2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt2, T_max=EPOCHS_FULL)

print(f'Phase 2 — {EPOCHS_FULL} epochs (fine-tune all)')
for ep in range(1, EPOCHS_FULL + 1):
    tl, ta = run_epoch(train_loader, opt2)
    vl, va = run_epoch(val_loader)
    sched2.step()
    hist['train_loss'].append(tl); hist['val_loss'].append(vl)
    hist['train_acc'].append(ta);  hist['val_acc'].append(va)
    if vl < best_val_loss:
        best_val_loss = vl
        torch.save(model, CKPT_DIR / 'best_full.pt')
    print(f'  Ep {ep:2d}/{EPOCHS_FULL} | loss {tl:.4f}/{vl:.4f} | acc {ta:.0%}/{va:.0%}')

## 8 — Training Curves

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
epochs_all = range(1, len(hist['train_loss']) + 1)
ax1.plot(epochs_all, hist['train_loss'], label='train')
ax1.plot(epochs_all, hist['val_loss'],   label='val')
ax1.axvline(EPOCHS_HEAD, color='gray', linestyle='--', alpha=0.5, label='phase boundary')
ax1.set_title('Loss'); ax1.legend()
ax2.plot(epochs_all, hist['train_acc'], label='train')
ax2.plot(epochs_all, hist['val_acc'],   label='val')
ax2.axvline(EPOCHS_HEAD, color='gray', linestyle='--', alpha=0.5)
ax2.set_title('Accuracy'); ax2.legend()
plt.tight_layout(); plt.show()

## 9 — Evaluation

In [ ]:
model.eval()
all_preds, all_true = [], []
with torch.no_grad():
    for imgs, targets in val_loader:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu().tolist()
        all_preds.extend(preds)
        all_true.extend(targets.tolist())

print(classification_report(all_true, all_preds, target_names=LABELS, zero_division=0))

In [ ]:
cm = confusion_matrix(all_true, all_preds, labels=list(range(len(LABELS))))
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS, ax=ax)
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix')
plt.tight_layout(); plt.show()

## 10 — บันทึก Model

In [ ]:
# โหลด checkpoint ที่ดีที่สุด แล้วบันทึกเป็น final
best_ckpt = CKPT_DIR / 'best_full.pt'
if not best_ckpt.exists():
    best_ckpt = CKPT_DIR / 'best_head.pt'

if best_ckpt.exists():
    best_model = torch.load(best_ckpt, map_location=DEVICE, weights_only=False)
    torch.save(best_model, MODEL_OUT)
    print(f'บันทึก best model → {MODEL_OUT}')
else:
    torch.save(model, MODEL_OUT)
    print(f'บันทึก model → {MODEL_OUT}')

## 11 — Quick Inference Test

In [ ]:
# ทดสอบ inference ด้วย vitro_vision.classifier
import importlib, sys
sys.path.insert(0, str(ROOT))

# reload เพื่อให้รับ model ใหม่
import vitro_vision.classifier as clf
clf._model = None
clf._model_loaded = False
importlib.reload(clf)

test_img = cv2.imread(all_paths[0])
status, conf = clf.predict(test_img)
true_label = all_labels[0]
print(f'ภาพ: {Path(all_paths[0]).name}')
print(f'True: {true_label} | Predicted: {status} ({conf:.1%})')